# R1：用 NumPy 手写逻辑回归

目标：实现二分类模型。它输出的不是连续数值，而是属于正类的概率：

$$z=Xw+b$$

$$p=\sigma(z)=\frac{1}{1+e^{-z}}$$

$$\operatorname{BCE}=-\frac1n\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]$$

In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

## 1. 二分类数据

特征 `x` 可以看成某项风险分数。低分为类别 `0`，高分为类别 `1`。

In [2]:
X = np.array([[0.5], [1.0], [1.5], [3.0], [3.5], [4.0]])
y = np.array([[0.0], [0.0], [0.0], [1.0], [1.0], [1.0]])

print('X.shape:', X.shape)
print('y.shape:', y.shape)
print('X =\n', X)
print('y =\n', y)

X.shape: (6, 1)
y.shape: (6, 1)
X =
 [[0.5]
 [1. ]
 [1.5]
 [3. ]
 [3.5]
 [4. ]]
y =
 [[0.]
 [0.]
 [0.]
 [1.]
 [1.]
 [1.]]


## 2. Sigmoid：将分数转换为概率

`z` 是线性分数，可以取任意值；sigmoid 将它压缩到 `(0, 1)`，因此可解释为概率。初始 `w=0,b=0` 时，所有样本的 `z=0`，预测概率都是 `0.5`。

In [3]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

w = np.zeros((1, 1))
b = 0.0

z = X @ w + b
p = sigmoid(z)

print('z =\n', z)
print('predicted probability p =\n', p)

z =
 [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
predicted probability p =
 [[0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]]


## 3. 二元交叉熵（binary cross-entropy, BCE）

`eps` 防止出现 `log(0)`。初始预测均为 `0.5`，因此初始 BCE 应约为 `0.6931`。

In [4]:
eps = 1e-12
bce_loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

print('initial BCE loss =', bce_loss)
assert np.isclose(bce_loss, np.log(2)), '初始 BCE 应接近 log(2) ≈ 0.6931。'

initial BCE loss = 0.6931471805579453


## 4. 梯度推导：为什么是 $p-y$

先看单个样本。设：

$$z=w^Tx+b$$

$$p=\sigma(z)=\frac{1}{1+e^{-z}}$$

$$\ell=-[y\log p+(1-y)\log(1-p)]$$

先对 $p$ 求导：

$$\frac{\partial \ell}{\partial p}=-\frac{y}{p}+\frac{1-y}{1-p}=\frac{p-y}{p(1-p)}$$

sigmoid 的导数为：

$$\frac{\partial p}{\partial z}=p(1-p)$$

根据链式法则：

$$\frac{\partial \ell}{\partial z}=\frac{\partial \ell}{\partial p}\frac{\partial p}{\partial z}=\frac{p-y}{p(1-p)}\cdot p(1-p)=p-y$$

最后，$\frac{\partial z}{\partial w}=x$，且 $\frac{\partial z}{\partial b}=1$，因此：

$$\frac{\partial \ell}{\partial w}=(p-y)x$$

$$\frac{\partial \ell}{\partial b}=p-y$$

一批 $n$ 个样本取平均后，得到代码中要用的矩阵形式：

$$\frac{\partial L}{\partial w}=\frac1nX^T(p-y)$$

$$\frac{\partial L}{\partial b}=\frac1n\sum_i(p_i-y_i)$$

这就是为什么逻辑回归的训练代码里，误差项写作 `error = p - y`。

## 5. 练习：计算一次梯度

初始时所有预测概率均为 $p=0.5$。请按刚才的批量公式计算 `error`、`dw`、`db`，然后进行一次更新。

预期：$dw=-0.625$，$db=0$；若学习率为 $0.1$，更新后 $w=0.0625$、$b=0$。

In [5]:
n = X.shape[0]
lr = 0.1

# TODO 1：error = p - y
error = p - y
# TODO 2：根据推导计算 dw 和 db
dw =(1/n)*X.T@(p-y)
db = np.mean(p-y)
# TODO 3：更新 w 和 b
w = w - lr*dw
b = b - lr*db
print('dw =', dw)
print('db =', db)
print('w after update =', w)
print('b after update =', b)

dw = [[-0.625]]
db = 0.0
w after update = [[0.0625]]
b after update = 0.0


## 6. 练习：逻辑回归训练循环

每轮按相同顺序执行：

$$X,w,b\rightarrow z\rightarrow p\rightarrow \mathrm{BCE}\rightarrow dw,db\rightarrow w,b$$

训练结束后，用阈值 $0.5$ 将概率变成类别：

$$\hat y=\mathbb{1}(p\ge0.5)$$

In [6]:
# 每次训练都从初始参数开始
w = np.zeros((1, 1))
b = 0.0

lr = 0.1
epochs = 1000
n = X.shape[0]
eps = 1e-12
loss_history = []

for epoch in range(epochs):
    # TODO 1：计算 z 和预测概率 p
    z = X @ w + b
    p = 1/(1+np.exp(-z))
    # TODO 2：计算 BCE loss
    loss = -np.mean(y*np.log(p+eps)+(1-y)*np.log(1-p+eps))
    # TODO 3：计算 error、dw、db
    error = p-y
    dw = (1/n)*X.T @(error)
    db = np.mean(error)
    # TODO 4：更新 w 和 b
    w = w - lr*dw
    b = b - lr*db
    # TODO 5：记录 loss，并每 100 轮打印一次
    loss_history.append(loss)
    if(epoch % 100 ==0):
        print(loss)
    
# TODO 6：训练结束后重新计算 p，生成 y_pred，并打印参数、概率、类别和准确率
z = X @ w + b
p = 1/(1+np.exp(-z))
y_pred = (p >= 0.5).astype(int)
accuracy = np.mean(y_pred == y)
print(f"final w ={w}")
print(f"final b ={b}",)
print("probabilities =\n", p)
print("predicted classes =\n", y_pred)
print("accuracy =", accuracy)

0.6931471805579453
0.32984154923676107
0.22300490917422403
0.16893391627065193
0.13663056453295006
0.11515773142046286
0.09982039837211432
0.08829173439872255
0.07929126715574114
0.07205672757332367
final w =[[2.5751]]
final b =-5.393893698564798
probabilities =
 [[0.0162]
 [0.0563]
 [0.1778]
 [0.9115]
 [0.9739]
 [0.9927]]
predicted classes =
 [[0]
 [0]
 [0]
 [1]
 [1]
 [1]]
accuracy = 1.0
